In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# --- SET YOUR PROJECT CONFIG HERE (Overrides .env) ---
os.environ["VERTEXAI_PROJECT"] = ""
os.environ["VERTEXAI_LOCATION"] = "us-central1"
os.environ["LLM_BACKEND"] = "vertex_ai"
os.environ["MODEL_NAME"] = "gemini-2.5-flash"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Task, Crew, Process

# Import local configurations and tools
from config.llm_config import get_llm
from tools.crewai_tools import (
    ParseDocxTool,
    ExtractProtocolTablesTool,
    DetectTemplateSignalsTool,
)

# Setup base directory (Jupyter-friendly)
base_dir = Path(os.getcwd())

# Load .env as fallback (if any other variables are needed)
load_dotenv(base_dir / ".env")

# Optional debug prints
print("--- Environment Variables ---")
print(f"LLM_BACKEND={os.getenv('LLM_BACKEND')}")
print(f"MODEL_NAME={os.getenv('MODEL_NAME')}")
print(f"VERTEXAI_PROJECT={os.getenv('VERTEXAI_PROJECT')}")

--- Environment Variables ---
LLM_BACKEND=vertex_ai
MODEL_NAME=gemini-2.5-flash
VERTEXAI_PROJECT=


In [2]:
def build_second_demo_crew() -> Crew:
    llm = get_llm()

    parse_docx_tool = ParseDocxTool()
    extract_tables_tool = ExtractProtocolTablesTool()
    detect_signals_tool = DetectTemplateSignalsTool()

    # 1. Define Agents (Removed max_rpm since Gemini Flash handles high load)
    document_parser_agent = Agent(
        role="Clinical Document Parser",
        goal="Parse protocol and SAP template documents into structured JSON artifacts.",
        backstory=(
            "You are meticulous about document structure, formatting, and reproducible parsing. "
            "You rely on your tools to process the files accurately."
        ),
        llm=llm,
        tools=[parse_docx_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=15,
    )

    signal_detection_agent = Agent(
        role="Template Signal Analyst",
        goal="Extract protocol tables and detect SAP template update/remove signals.",
        backstory=(
            "You specialize in converting parsed clinical documents into structured workflow artifacts, "
            "including table extractions and update-unit lists."
        ),
        llm=llm,
        tools=[extract_tables_tool, detect_signals_tool],
        verbose=True,
        allow_delegation=False,
        max_rpm=15,
    )

    workflow_reporter_agent = Agent(
        role="Workflow Reporter",
        goal="Summarize the generated artifacts and explain what the demo accomplished.",
        backstory=(
            "You create concise, accurate summaries of automation outputs for training and demonstration."
        ),
        llm=llm,
        verbose=True,
        allow_delegation=False,
        max_rpm=15,
    )

    # 2. Define Tasks (Using EXACT argument names for Pydantic schema validation)
    parse_protocol_task = Task(
        description=(
            "Use the parse_docx_tool to parse the protocol DOCX file.\n"
            "- file_path: {protocol_path}\n"
            "- output_json_path: {protocol_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed protocol file.",
        agent=document_parser_agent,
    )

    parse_template_task = Task(
        description=(
            "Use the parse_docx_tool to parse the SAP template DOCX file.\n"
            "- file_path: {template_path}\n"
            "- output_json_path: {template_json_path}\n"
            "Return a short confirmation that includes paragraph count and table count."
        ),
        expected_output="A short confirmation JSON-like summary for the parsed SAP template file.",
        agent=document_parser_agent,
    )

    extract_tables_task = Task(
        description=(
            "Use the extract_protocol_tables_tool on the parsed protocol JSON.\n"
            "- parsed_protocol_json_path: {protocol_json_path}\n"
            "- output_json_path: {protocol_tables_json_path}\n"
            "Return a short confirmation with the number of extracted tables."
        ),
        expected_output="A short confirmation JSON-like summary for extracted protocol tables.",
        agent=signal_detection_agent,
        context=[parse_protocol_task],
    )

    detect_signals_task = Task(
        description=(
            "Use the detect_template_signals_tool on the parsed template JSON.\n"
            "- parsed_template_json_path: {template_json_path}\n"
            "- output_json_path: {update_units_json_path}\n"
            "Current signal rules: blue text = update placeholder, green text = guidance/remove.\n"
            "Return a short confirmation with the number of detected update units."
        ),
        expected_output="A short confirmation JSON-like summary for detected template update units.",
        agent=signal_detection_agent,
        context=[parse_template_task],
    )

    summarize_task = Task(
        description=(
            "Summarize what artifacts were created in this second demo. "
            "Use the prior task results and produce a concise operational summary "
            "that explains what the CrewAI version accomplished."
        ),
        expected_output=(
            "A concise summary of the generated outputs, counts, and the purpose of this second demo."
        ),
        agent=workflow_reporter_agent,
        context=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
        ],
    )

    # 3. Assemble the Crew
    return Crew(
        agents=[
            document_parser_agent,
            signal_detection_agent,
            workflow_reporter_agent,
        ],
        tasks=[
            parse_protocol_task,
            parse_template_task,
            extract_tables_task,
            detect_signals_task,
            summarize_task,
        ],
        process=Process.sequential,
        verbose=True,
        max_rpm=15,
    )

In [3]:
print("--- Initializing CrewAI Execution ---")

data_dir = base_dir / "data"
out_dir = base_dir / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
inputs = {
    "protocol_path": str(data_dir / "protocol.docx"),
    "template_path": str(data_dir / "sap_template.docx"),
    "protocol_json_path": str(out_dir / "protocol_parsed.json"),
    "template_json_path": str(out_dir / "sap_template_parsed.json"),
    "protocol_tables_json_path": str(out_dir / "protocol_tables.json"),
    "update_units_json_path": str(out_dir / "update_units.json"),
}

# Build and execute the crew
crew = build_second_demo_crew()
result = crew.kickoff(inputs=inputs)

# Save the LLM summary result
summary_path = out_dir / "crewai_demo2_result.txt"
summary_path.write_text(str(result), encoding="utf-8")

# Create and save a deterministic payload summary
deterministic_summary = {
    "protocol_json_path": inputs["protocol_json_path"],
    "template_json_path": inputs["template_json_path"],
    "protocol_tables_json_path": inputs["protocol_tables_json_path"],
    "update_units_json_path": inputs["update_units_json_path"],
    "crewai_result_path": str(summary_path),
}

(out_dir / "crewai_demo2_summary.json").write_text(
    json.dumps(deterministic_summary, indent=2),
    encoding="utf-8",
)

print("\n--- CrewAI Second Demo Completed ---")
print(f"Result summary saved to: {summary_path}")

--- Initializing CrewAI Execution ---


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8dadcee2-7ce4-4cdd-a0ad-c5ea2bfc2a00                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Clinical Document Parser                                                                                │
│                                                                                                                 │
│  Thought: Action: parse_docx_tool                                                                               │
│                                                                                                                 │
│  Using Tool: parse_docx_tool                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Workflow Reporter                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The CrewAI version successfully parsed two source documents: `protocol.docx` and `sap_template.docx`. This     │
│  process generated `protocol_parsed.json` containing 218 paragraphs and 1 table, and                            │
│  `sap_template_parsed.json` with 109 paragraphs and 1 table. Additionally, the table from the parsed protocol   │
│  was specifically extracted into `protocol_tables.json`. The core accomplishment of this demo was the           │
│  successful detection of 19 "update units" from the processed content, demonstrating an automated capability    │
│  to extract and identify specific data points for potential updates or analysis from diverse document types.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 5738b6d5-4206-455d-a7f5-674c35634c2e                                                                     │
│  Agent: Workflow Reporter                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8dadcee2-7ce4-4cdd-a0ad-c5ea2bfc2a00                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: The CrewAI version successfully parsed two source documents: `protocol.docx` and                 │
│  `sap_template.docx`. This process generated `protocol_parsed.json` containing 218 paragraphs and 1 table, and  │
│  `sap_template_parsed.json` with 109 paragraphs and 1 table. Additionally, the table from the parsed protocol   │
│  was specifically extracted into `protocol_tables.json`. The core accomplishment of this demo was the           │
│  successful detection of 19 "update units" from the processed content, demonstrating an automated capability    │
│  to extract and identify specific data points for potential updates or analysis from diverse document types.    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RecursionError: maximum recursion depth exceeded